In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import dash
from dash import Dash, dcc, html, Input, Output, no_update

df = pd.read_csv("Accelerometer_XYZ_Combined.csv")

df["time"] = pd.to_datetime(df["time"])

df = df.dropna(subset=["x", "y", "z"], how="all")

df[["x", "y", "z"]] = df[["x", "y", "z"]].interpolate()

In [4]:
df = df.reset_index(drop=True)

def smooth_extend_data(
    new_data,
    x_col="time",
    y_cols=("x", "y", "z"),
    trace_indices=(0, 1, 2),
    max_points=200
):

    if new_data.empty:
        return no_update

    times = new_data[x_col].tolist()

    y_values = [
        new_data[col].tolist()
        for col in y_cols
    ]

    x_values = [
        times for _ in y_cols
    ]

    update_data = {
        "x": x_values,
        "y": y_values
    }

    return update_data, list(trace_indices), max_points

app = Dash(__name__)

app.layout = html.Div([
    
    html.H1("Live Smartphone Accelerometer"),

    dcc.Graph(
        id="accelerometer-graph",
        figure=go.Figure(
            data=[
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines",
                    name="X"
                ),
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines",
                    name="Y"
                ),
                go.Scatter(
                    x=[],
                    y=[],
                    mode="lines",
                    name="Z"
                )
            ]
        )
    ),

    dcc.Interval(
        id="update-interval",
        interval=200,
        n_intervals=0
    )
])

current_index = 0
batch_size = 5


@app.callback(
    Output("accelerometer-graph", "extendData"),
    Input("update-interval", "n_intervals")
)
def update_graph(n_intervals):

    global current_index

    if current_index >= len(df):
        return no_update

    new_data = df.iloc[
        current_index:current_index + batch_size
    ]

    current_index += batch_size

    return smooth_extend_data(
        new_data,
        x_col="time",
        y_cols=("x", "y", "z"),
        trace_indices=(0, 1, 2),
        max_points=200
    )

app.run(
    debug=False,
    jupyter_mode="external"
)

Dash app running on http://127.0.0.1:8050/
